# Parquet Wrangling Playground

This notebook is for traditional data understanding and wrangling against the processed Parquet outputs.

The goal is to help you answer questions like:

- what fields are actually available in each dataset
- which business keys are reliable enough to use in a final unified file
- what should stay source-specific versus what can be safely rolled up to a customer-level file

Important guardrail: this notebook does **not** force a direct merge between vend and consumption. Instead, it keeps `Consumer Master` as the anchor table and brings in separate aggregate summaries from the other sources.

In [1]:
from __future__ import annotations

# Standard library imports for path handling and repo-local imports.
from pathlib import Path
import sys

# Core wrangling libraries used in the notebook.
import pandas as pd
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    # Walk upward until we find the repo structure expected by the notebook.
    start_path = (start or Path.cwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "src").exists() and (candidate / "config").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing src/ and config/.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: F:\Secure\CashFlowMgmt


In [2]:
# Point to the processed-data folder so this notebook works without the raw CSVs.
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

consumer_master_path = PROCESSED_DIR / "consumer_master.parquet"
consumption_path = PROCESSED_DIR / "consumption.parquet"
vend_path = PROCESSED_DIR / "vend.parquet"

for path in [consumer_master_path, consumption_path, vend_path]:
    print(path.name, "exists" if path.exists() else "missing", path)

# Load each processed dataset as-is so we can inspect the true post-ingestion structure.
consumer_master_df = pd.read_parquet(consumer_master_path)
consumption_df = pd.read_parquet(consumption_path)
vend_df = pd.read_parquet(vend_path)

consumer_master.parquet exists F:\Secure\CashFlowMgmt\data\processed\consumer_master.parquet
consumption.parquet exists F:\Secure\CashFlowMgmt\data\processed\consumption.parquet
vend.parquet exists F:\Secure\CashFlowMgmt\data\processed\vend.parquet


In [3]:
# Small helper functions keep the notebook readable and make the wrangling flow reusable.

KEY_COLUMNS = ["consumernumber_normalized", "meterno_normalized"]
IMPORTANT_MASTER_FIELDS = [
    "consumernumber",
    "meterno",
    "accountingmode",
    "connectionstatus",
    "tariffcode",
    "area_type",
    "feedercode",
    "dtcode",
    "substationname",
    "meterbalance",
    "balanceupdatedon",
]


def available_columns(df: pd.DataFrame, candidates: list[str]) -> list[str]:
    return [column for column in candidates if column in df.columns]


def frame_overview(df: pd.DataFrame, name: str) -> pd.DataFrame:
    overview = {
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "memory_mb": round(df.memory_usage(deep=True).sum() / (1024 ** 2), 2),
    }
    for key in KEY_COLUMNS:
        if key in df.columns:
            overview[f"non_null_{key}"] = int(df[key].notna().sum())
            overview[f"distinct_{key}"] = int(df[key].dropna().nunique())
    return pd.DataFrame([overview])


def missing_summary(df: pd.DataFrame, top_n: int = 20) -> pd.DataFrame:
    summary = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum().values,
    })
    summary["missing_pct"] = (summary["missing_count"] / len(df) * 100).round(2)
    return summary.sort_values(["missing_pct", "missing_count"], ascending=False).head(top_n)


def duplicate_key_summary(df: pd.DataFrame, key: str) -> pd.DataFrame:
    if key not in df.columns:
        return pd.DataFrame([{"key": key, "rows": 0, "distinct_keys": 0, "duplicate_keys": 0, "note": "column not available"}])
    non_null = df[df[key].notna()].copy()
    counts = non_null[key].value_counts()
    return pd.DataFrame([{
        "key": key,
        "rows": len(non_null),
        "distinct_keys": int(non_null[key].nunique()),
        "duplicate_keys": int((counts > 1).sum()),
        "note": "",
    }])


def key_overlap(left_df: pd.DataFrame, right_df: pd.DataFrame, key: str, left_name: str, right_name: str) -> pd.DataFrame:
    if key not in left_df.columns or key not in right_df.columns:
        return pd.DataFrame([{"key": key, "left": left_name, "right": right_name, "overlap_pct_of_left": None}])
    left_keys = set(left_df[key].dropna().astype(str))
    right_keys = set(right_df[key].dropna().astype(str))
    overlap = left_keys & right_keys
    overlap_pct = round(len(overlap) / len(left_keys) * 100, 2) if left_keys else None
    return pd.DataFrame([{
        "key": key,
        "left": left_name,
        "right": right_name,
        "left_distinct_keys": len(left_keys),
        "right_distinct_keys": len(right_keys),
        "overlap_keys": len(overlap),
        "overlap_pct_of_left": overlap_pct,
    }])

## 1. High-level shape check

Start with row counts, column counts, memory footprint, and normalized-key availability before doing any deeper wrangling.

In [4]:
dataset_overview = pd.concat(
    [
        frame_overview(consumer_master_df, "consumer_master"),
        frame_overview(vend_df, "vend"),
        frame_overview(consumption_df, "consumption"),
    ],
    ignore_index=True,
)

display(dataset_overview)

,dataset,rows,columns,memory_mb,non_null_consumernumber_normalized,distinct_consumernumber_normalized,non_null_meterno_normalized,distinct_meterno_normalized
0,consumer_master,1548361,38,2626.84,1548361,1548361,987433,987408
1,vend,1503445,19,1061.59,1503445,639662,1503445,639759
2,consumption,580860,25,595.43,580860,19270,580860,19270


## 2. Column inventory and sample rows

This is the fastest way to understand what is really in each Parquet file after ingestion and normalization.

In [5]:
for dataset_name, df in {
    "consumer_master": consumer_master_df,
    "vend": vend_df,
    "consumption": consumption_df,
}.items():
    print(f"\n--- {dataset_name.upper()} COLUMNS ({len(df.columns)}) ---")
    print(df.columns.tolist())
    display(df.head(3))


--- CONSUMER_MASTER COLUMNS (38) ---
['consumername', 'consumernumber', 'meterno', 'feedercode', 'dtcode', 'metertype', 'sanctionedload', 'privilagedconsumer', 'netmetering', 'tariffcode', 'accountingmode', 'connectionstatus', 'meterinstallationdate', 'meterbalance', 'balanceupdatedon', 'kno', 'meterid', 'meter_serial_no', 'gaaid', 'gis_latitude', 'gis_longitude', 'sanctionedloadunit', 'connection_type', 'area_type', 'billingunit', 'isnetmetering', 'contract_demand', 'consumernumber_normalized', 'meterno_normalized', 'source_file', 'meterinstallationdate_raw', 'meterinstallationdate_parsed', 'meterinstallationdate_parse_status', 'balanceupdatedon_raw', 'balanceupdatedon_parsed', 'balanceupdatedon_parse_status', 'has_valid_gis', 'has_feeder_dt']


,consumername,consumernumber,meterno,feedercode,dtcode,metertype,sanctionedload,privilagedconsumer,netmetering,tariffcode,...,meterno_normalized,source_file,meterinstallationdate_raw,meterinstallationdate_parsed,meterinstallationdate_parse_status,balanceupdatedon_raw,balanceupdatedon_parsed,balanceupdatedon_parse_status,has_valid_gis,has_feeder_dt
0,MUKESH MANDASL,1231053137,NB11887278,FD426,DT09172,Single phase,0.25,No,No,KJ,...,NB11887278,ConsumerMaster20260401.csv,11-Mar-2025,2025-03-11,date_only,25-Mar-26 00:00,2026-03-25,parsed_datetime,True,True
1,VIJAY BHUSHAN SAH,122410808173,NB11916418,<NA>,<NA>,Single phase,1.0,No,No,DS1D,...,NB11916418,ConsumerMaster20260401.csv,28-Apr-2025,2025-04-28,date_only,06-Oct-25 00:00,2025-10-06,parsed_datetime,True,False
2,NITISH KUMAR,123111331382,NB11525680,<NA>,<NA>,Single phase,1.0 kW,No,No,DS1D,...,NB11525680,ConsumerMaster20260401.csv,16-Jan-2025,2025-01-16,date_only,25-Mar-26 00:00,2026-03-25,parsed_datetime,<NA>,False



--- VEND COLUMNS (19) ---
['consumernumber', 'meterno', 'transactionamount', 'issuedate', 'consumernumber_normalized', 'meterno_normalized', 'source_file', 'issuedate_raw', 'issuedate_parse_status', 'issuedate_is_time_only', 'issuedate_parsed', 'issuedate_time_only_text', 'issuedate_time_hour', 'vend_date', 'vend_month', 'vend_week', 'vend_day_of_week', 'vend_hour', 'analysis_hour']


,consumernumber,meterno,transactionamount,issuedate,consumernumber_normalized,meterno_normalized,source_file,issuedate_raw,issuedate_parse_status,issuedate_is_time_only,issuedate_parsed,issuedate_time_only_text,issuedate_time_hour,vend_date,vend_month,vend_week,vend_day_of_week,vend_hour,analysis_hour
0,13010125364,NB11170869,300.0,2026-01-01 00:08:15.701,13010125364,NB11170869,VendData20260401.csv,2026-01-01 00:08:15.701,parsed_datetime,False,2026-01-01 00:08:15.701,<NA>,<NA>,2026-01-01,2026-01,1,Thursday,0,0
1,118301362146,NB30004318,5000.0,2026-01-01 00:09:04.986,118301362146,NB30004318,VendData20260401.csv,2026-01-01 00:09:04.986,parsed_datetime,False,2026-01-01 00:09:04.986,<NA>,<NA>,2026-01-01,2026-01,1,Thursday,0,0
2,119309790164,NB10919580,100.0,2026-01-01 00:09:05.377,119309790164,NB10919580,VendData20260401.csv,2026-01-01 00:09:05.377,parsed_datetime,False,2026-01-01 00:09:05.377,<NA>,<NA>,2026-01-01,2026-01,1,Thursday,0,0



--- CONSUMPTION COLUMNS (25) ---
['substationname', 'consumernumber', 'meterno', 'feedercode', 'dtcode', 'metertype', 'supplyvoltage', 'connectionstatus', 'meterinstallationdate', 'midnightdate', 'kwh_abs', 'kvah_abs', 'kwh_consumption', 'kvah_consumption', 'consumernumber_normalized', 'meterno_normalized', 'source_file', 'midnightdate_raw', 'midnightdate_parsed', 'midnightdate_parse_status', 'midnightdate_parse_success', 'date', 'month', 'week', 'day_of_week']


,substationname,consumernumber,meterno,feedercode,dtcode,metertype,supplyvoltage,connectionstatus,meterinstallationdate,midnightdate,...,meterno_normalized,source_file,midnightdate_raw,midnightdate_parsed,midnightdate_parse_status,midnightdate_parse_success,date,month,week,day_of_week
0,33/11 KV Jagdishpur PSS,40351,NB50004037,FD470,40351,LT CT,11,Regular,25-Oct-2024,2026-01-01 00:00:00,...,NB50004037,ConsumptionData20260401.csv,2026-01-01 00:00:00,2026-01-01,parsed_datetime,True,2026-01-01,2026-01,1,Thursday
1,33/11 KV Jagdishpur PSS,40351,NB50004037,FD470,40351,LT CT,11,Regular,25-Oct-2024,2026-01-02 00:00:00,...,NB50004037,ConsumptionData20260401.csv,2026-01-02 00:00:00,2026-01-02,parsed_datetime,True,2026-01-02,2026-01,1,Friday
2,33/11 KV Jagdishpur PSS,40351,NB50004037,FD470,40351,LT CT,11,Regular,25-Oct-2024,2026-01-03 00:00:00,...,NB50004037,ConsumptionData20260401.csv,2026-01-03 00:00:00,2026-01-03,parsed_datetime,True,2026-01-03,2026-01,1,Saturday


## 3. Missingness and duplicate-key checks

These checks help you decide which fields are strong enough to carry into a future combined file and which ones need caveats.

In [6]:
display(missing_summary(consumer_master_df, top_n=20))
display(missing_summary(vend_df, top_n=20))
display(missing_summary(consumption_df, top_n=20))

duplicate_checks = pd.concat(
    [
        duplicate_key_summary(consumer_master_df, "consumernumber_normalized"),
        duplicate_key_summary(consumer_master_df, "meterno_normalized"),
        duplicate_key_summary(vend_df, "consumernumber_normalized"),
        duplicate_key_summary(vend_df, "meterno_normalized"),
        duplicate_key_summary(consumption_df, "consumernumber_normalized"),
        duplicate_key_summary(consumption_df, "meterno_normalized"),
    ],
    ignore_index=True,
)

display(duplicate_checks)

,column,missing_count,missing_pct
4,dtcode,843240,54.46
3,feedercode,843077,54.45
13,meterbalance,562704,36.34
14,balanceupdatedon,562386,36.32
33,balanceupdatedon_raw,562386,36.32
34,balanceupdatedon_parsed,562386,36.32
12,meterinstallationdate,560929,36.23
30,meterinstallationdate_raw,560929,36.23
31,meterinstallationdate_parsed,560929,36.23
2,meterno,560928,36.23


,column,missing_count,missing_pct
11,issuedate_time_only_text,1503445,100.0
12,issuedate_time_hour,1503445,100.0
0,consumernumber,0,0.0
1,meterno,0,0.0
2,transactionamount,0,0.0
3,issuedate,0,0.0
4,consumernumber_normalized,0,0.0
5,meterno_normalized,0,0.0
6,source_file,0,0.0
7,issuedate_raw,0,0.0


,column,missing_count,missing_pct
3,feedercode,28620,4.93
6,supplyvoltage,28620,4.93
4,dtcode,28589,4.92
12,kwh_consumption,958,0.16
13,kvah_consumption,958,0.16
0,substationname,0,0.00
1,consumernumber,0,0.00
2,meterno,0,0.00
5,metertype,0,0.00
7,connectionstatus,0,0.00


,key,rows,distinct_keys,duplicate_keys,note
0,consumernumber_normalized,1548361,1548361,0,
1,meterno_normalized,987433,987408,24,
2,consumernumber_normalized,1503445,639662,314175,
3,meterno_normalized,1503445,639759,314191,
4,consumernumber_normalized,580860,19270,19157,
5,meterno_normalized,580860,19270,19157,


## 4. Key overlap across sources

This section is useful before designing a single-file output, because it shows how much of each source can be aligned back to the master by normalized consumer or meter keys.

In [7]:
key_overlap_summary = pd.concat(
    [
        key_overlap(vend_df, consumer_master_df, "consumernumber_normalized", "vend", "consumer_master"),
        key_overlap(vend_df, consumer_master_df, "meterno_normalized", "vend", "consumer_master"),
        key_overlap(consumption_df, consumer_master_df, "consumernumber_normalized", "consumption", "consumer_master"),
        key_overlap(consumption_df, consumer_master_df, "meterno_normalized", "consumption", "consumer_master"),
    ],
    ignore_index=True,
)

display(key_overlap_summary)

,key,left,right,left_distinct_keys,right_distinct_keys,overlap_keys,overlap_pct_of_left
0,consumernumber_normalized,vend,consumer_master,639662,1548361,463277,72.43
1,meterno_normalized,vend,consumer_master,639759,987408,288292,45.06
2,consumernumber_normalized,consumption,consumer_master,19270,1548361,0,0.00
3,meterno_normalized,consumption,consumer_master,19270,987408,1,0.01


## 5. Source-specific aggregates to support a future unified file

The safest path is to build independent summaries from vend and consumption, then attach those summaries to the master table. This avoids pretending the two activity sources line up perfectly with each other.

In [8]:
# Convert numeric fields defensively. The exact source columns may vary over time,
# so we only use fields that are present in the processed extracts.
vend_work = vend_df.copy()
consumption_work = consumption_df.copy()

if "transactionamount" in vend_work.columns:
    vend_work["transactionamount_numeric"] = pd.to_numeric(vend_work["transactionamount"], errors="coerce")

for column in ["kwh", "kvah", "daily_kwh", "daily_kvah", "consumption_kwh", "consumption_kvah"]:
    if column in consumption_work.columns:
        consumption_work[column] = pd.to_numeric(consumption_work[column], errors="coerce")

consumption_date_column = "midnightdate" if "midnightdate" in consumption_work.columns else None
if consumption_date_column is not None:
    consumption_work[consumption_date_column] = pd.to_datetime(consumption_work[consumption_date_column], errors="coerce")

vend_agg_by_consumer = (
    vend_work[vend_work["consumernumber_normalized"].notna()]
    .groupby("consumernumber_normalized", dropna=True)
    .agg(
        vend_txn_count_consumer=("consumernumber_normalized", "size"),
        vend_amount_sum_consumer=("transactionamount_numeric", "sum"),
        vend_amount_mean_consumer=("transactionamount_numeric", "mean"),
        vend_amount_median_consumer=("transactionamount_numeric", "median"),
    )
    .reset_index()
)

vend_agg_by_meter = (
    vend_work[vend_work["meterno_normalized"].notna()]
    .groupby("meterno_normalized", dropna=True)
    .agg(
        vend_txn_count_meter=("meterno_normalized", "size"),
        vend_amount_sum_meter=("transactionamount_numeric", "sum"),
        vend_amount_mean_meter=("transactionamount_numeric", "mean"),
        vend_amount_median_meter=("transactionamount_numeric", "median"),
    )
    .reset_index()
)

consumption_value_columns = [column for column in ["kwh", "kvah", "daily_kwh", "daily_kvah", "consumption_kwh", "consumption_kvah"] if column in consumption_work.columns]

consumption_agg_spec_consumer = {"consumption_row_count_consumer": ("consumernumber_normalized", "size")}
consumption_agg_spec_meter = {"consumption_row_count_meter": ("meterno_normalized", "size")}

for column in consumption_value_columns:
    consumption_agg_spec_consumer[f"avg_{column}_consumer"] = (column, "mean")
    consumption_agg_spec_consumer[f"max_{column}_consumer"] = (column, "max")
    consumption_agg_spec_meter[f"avg_{column}_meter"] = (column, "mean")
    consumption_agg_spec_meter[f"max_{column}_meter"] = (column, "max")

if consumption_date_column is not None:
    consumption_agg_spec_consumer["latest_consumption_date_consumer"] = (consumption_date_column, "max")
    consumption_agg_spec_meter["latest_consumption_date_meter"] = (consumption_date_column, "max")

consumption_agg_by_consumer = (
    consumption_work[consumption_work["consumernumber_normalized"].notna()]
    .groupby("consumernumber_normalized", dropna=True)
    .agg(**consumption_agg_spec_consumer)
    .reset_index()
)

consumption_agg_by_meter = (
    consumption_work[consumption_work["meterno_normalized"].notna()]
    .groupby("meterno_normalized", dropna=True)
    .agg(**consumption_agg_spec_meter)
    .reset_index()
)

display(vend_agg_by_consumer.head())
display(vend_agg_by_meter.head())
display(consumption_agg_by_consumer.head())
display(consumption_agg_by_meter.head())

,consumernumber_normalized,vend_txn_count_consumer,vend_amount_sum_consumer,vend_amount_mean_consumer,vend_amount_median_consumer
0,10661125000074,1,1.74,1.74,1.74
1,10661125000082,4,941.81,235.4525,300.0
2,10661125000120,15,317.0,21.133333,20.0
3,10661125000147,2,51.14,25.57,25.57
4,10661125000228,1,200.0,200.0,200.0


,meterno_normalized,vend_txn_count_meter,vend_amount_sum_meter,vend_amount_mean_meter,vend_amount_median_meter
0,NB10000002,3,5298.61,1766.203333,298.0
1,NB10000007,2,400.0,200.0,200.0
2,NB10000011,3,305.4,101.8,100.0
3,NB10000012,1,1500.0,1500.0,1500.0
4,NB10000021,17,1300.45,76.497059,65.0


,consumernumber_normalized,consumption_row_count_consumer,latest_consumption_date_consumer
0,40351,31,2026-01-31
1,40352,31,2026-01-31
2,40353,31,2026-01-31
3,40354,31,2026-01-31
4,40355n0,17,2026-01-31


,meterno_normalized,consumption_row_count_meter,latest_consumption_date_meter
0,17060930,31,2026-01-31
1,17061013,31,2026-01-31
2,17061073,31,2026-01-31
3,17096188,31,2026-01-31
4,17096227,31,2026-01-31


## 6. Draft master-anchored unified file

This draft is deliberately conservative:

- it keeps the original master fields
- it brings in vend summaries by consumer key and by meter key separately
- it brings in consumption summaries by consumer key and by meter key separately
- it does **not** claim that vend and consumption are directly reconciled to each other

That gives you a practical starting point for deciding which fields belong in the final single-file design.

In [9]:
draft_unified = consumer_master_df.copy()

# Keep a compact business-friendly subset first if you want a lighter working file.
# Comment this block out if you prefer to keep every master column in the draft.
preferred_master_columns = available_columns(
    draft_unified,
    IMPORTANT_MASTER_FIELDS + KEY_COLUMNS,
)
if preferred_master_columns:
    draft_unified = draft_unified[preferred_master_columns].copy()

draft_unified = draft_unified.merge(
    vend_agg_by_consumer,
    on="consumernumber_normalized",
    how="left",
)
draft_unified = draft_unified.merge(
    vend_agg_by_meter,
    on="meterno_normalized",
    how="left",
)
draft_unified = draft_unified.merge(
    consumption_agg_by_consumer,
    on="consumernumber_normalized",
    how="left",
)
draft_unified = draft_unified.merge(
    consumption_agg_by_meter,
    on="meterno_normalized",
    how="left",
)

# A few simple indicator fields make it easier to review coverage in the draft output.
draft_unified["has_vend_summary_consumer"] = draft_unified.get("vend_txn_count_consumer", pd.Series(0, index=draft_unified.index)).fillna(0).gt(0)
draft_unified["has_vend_summary_meter"] = draft_unified.get("vend_txn_count_meter", pd.Series(0, index=draft_unified.index)).fillna(0).gt(0)
draft_unified["has_consumption_summary_consumer"] = draft_unified.get("consumption_row_count_consumer", pd.Series(0, index=draft_unified.index)).fillna(0).gt(0)
draft_unified["has_consumption_summary_meter"] = draft_unified.get("consumption_row_count_meter", pd.Series(0, index=draft_unified.index)).fillna(0).gt(0)

display(draft_unified.head())
display(pd.DataFrame([{
    "rows": len(draft_unified),
    "columns": len(draft_unified.columns),
    "rows_with_vend_consumer_summary": int(draft_unified["has_vend_summary_consumer"].sum()),
    "rows_with_vend_meter_summary": int(draft_unified["has_vend_summary_meter"].sum()),
    "rows_with_consumption_consumer_summary": int(draft_unified["has_consumption_summary_consumer"].sum()),
    "rows_with_consumption_meter_summary": int(draft_unified["has_consumption_summary_meter"].sum()),
}]))

,consumernumber,meterno,accountingmode,connectionstatus,tariffcode,area_type,feedercode,dtcode,meterbalance,balanceupdatedon,...,vend_amount_mean_meter,vend_amount_median_meter,consumption_row_count_consumer,latest_consumption_date_consumer,consumption_row_count_meter,latest_consumption_date_meter,has_vend_summary_consumer,has_vend_summary_meter,has_consumption_summary_consumer,has_consumption_summary_meter
0,1231053137,NB11887278,Prepaid,Regular,KJ,Rural,FD426,DT09172,1999.18,25-Mar-26 00:00,...,<NA>,<NA>,NaN,NaT,NaN,NaT,False,False,False,False
1,122410808173,NB11916418,Prepaid,Regular,DS1D,Rural,<NA>,<NA>,179.068,06-Oct-25 00:00,...,<NA>,<NA>,NaN,NaT,NaN,NaT,False,False,False,False
2,123111331382,NB11525680,Prepaid,Regular,DS1D,<NA>,<NA>,<NA>,57.379,25-Mar-26 00:00,...,<NA>,<NA>,NaN,NaT,NaN,NaT,False,False,False,False
3,120206254795,NB10837132,Prepaid,Regular,DS1D,<NA>,FD093,DT01967,476.601,25-Mar-26 00:00,...,<NA>,<NA>,NaN,NaT,NaN,NaT,False,False,False,False
4,1203044679,NB11695922,Prepaid,Regular,DS1D,<NA>,<NA>,<NA>,436.667,25-Mar-26 00:00,...,<NA>,<NA>,NaN,NaT,NaN,NaT,False,False,False,False


,rows,columns,rows_with_vend_consumer_summary,rows_with_vend_meter_summary,rows_with_consumption_consumer_summary,rows_with_consumption_meter_summary
0,1548361,28,463277,288302,0,1


## 7. Save or iterate

Leave the export commented out until you are comfortable with the shape of the draft file. Once the field list feels right, this cell can become the start of your final single-file pipeline.

In [10]:
# Suggested next steps:
# 1. review draft_unified.columns.tolist()
# 2. decide which source-specific summary fields should stay in the final file
# 3. add business rules for low-value columns, stale fields, or duplicate-key edge cases
# 4. only then write the draft to disk

# Example export path:
# output_path = PROJECT_ROOT / "data" / "processed" / "draft_unified_customer_file.parquet"
# draft_unified.to_parquet(output_path, index=False)
# print(f"Saved draft file to {output_path}")